# R03b - Analisis del espacio latente

Pregunta objetivo: ?las zonas de transicion siguen siendo distinguibles en `mu`?

El notebook recorre `val+test` completos para calcular densidad de transicion, porosidad y correlaciones por canal. Para PCA + UMAP usa una muestra estratificada y acotada por clase para mantener el analisis interactivo.

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import umap
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

from poregen.analysis import (
    build_patch_loader,
    encode_mu,
    flatten_mu,
    load_r03_runtime,
    mask_gradient_density,
    mean_latent_channels,
    transition_percentile_threshold,
)
from poregen.training import seed_everything


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "src" / "poregen").exists():
            return candidate
    raise FileNotFoundError("Could not infer the repo root from the current working directory.")


REPO_ROOT = find_repo_root()
CHECKPOINT_PATH = REPO_ROOT / "runs" / "vae" / "r03" / "last.ckpt"
CONFIG_PATH = REPO_ROOT / "src" / "poregen" / "configs" / "r03.yaml"
DATA_ROOT = None
OUTPUT_DIR = REPO_ROOT / "runs" / "analysis" / "r03b_latent_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INCLUDE_HIGH_POROSITY_CLASS = True
MAX_SAMPLES_PER_CLASS = 1024
KNN_FOLDS = 5
UMAP_NEIGHBORS = 30
UMAP_MIN_DIST = 0.1

seed_everything(42)
rng = np.random.default_rng(42)

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        f"Checkpoint not found: {CHECKPOINT_PATH}. Update CHECKPOINT_PATH once the R03 run finishes."
    )


In [ ]:
runtime = load_r03_runtime(
    CHECKPOINT_PATH,
    config_path=CONFIG_PATH,
    data_root=DATA_ROOT,
    repo_root=REPO_ROOT,
)

analysis_batch_size = min(64, int(runtime.cfg["data"].get("batch_size", 64)))
val_loader = build_patch_loader(runtime.cfg, runtime.data_root, "val", batch_size=analysis_batch_size, shuffle=False)
test_loader = build_patch_loader(runtime.cfg, runtime.data_root, "test", batch_size=analysis_batch_size, shuffle=False)

print(f"Loaded R03 checkpoint at step {runtime.checkpoint_step} on {runtime.device}.")
print(f"Data root: {runtime.data_root}")
print(f"Val / test patches: {len(val_loader.dataset):,} / {len(test_loader.dataset):,}")


In [ ]:
def collect_patch_annotations(loaders: dict[str, object]) -> pd.DataFrame:
    rows = []
    for split_name, loader in loaders.items():
        dataset_index = 0
        for batch in tqdm(loader, desc=f"{split_name} annotations"):
            transition_density = mask_gradient_density(batch["mask"]).cpu().numpy()
            porosity = batch["mask"].float().mean(dim=(1, 2, 3, 4)).cpu().numpy()
            batch_size = len(batch["volume_id"])
            for i in range(batch_size):
                rows.append({
                    "split": split_name,
                    "dataset_index": dataset_index,
                    "volume_id": batch["volume_id"][i],
                    "transition_density": float(transition_density[i]),
                    "porosity": float(porosity[i]),
                })
                dataset_index += 1
    return pd.DataFrame(rows)


annotations = collect_patch_annotations({"val": val_loader, "test": test_loader})
transition_threshold = transition_percentile_threshold(torch.tensor(annotations["transition_density"].values), percentile=75.0)
phi_threshold = float(annotations["porosity"].quantile(0.75))


def assign_patch_class(transition_density: float, porosity: float) -> str:
    if transition_density >= transition_threshold:
        return "high_transition"
    if INCLUDE_HIGH_POROSITY_CLASS and porosity >= phi_threshold:
        return "high_porosity"
    return "interior"


annotations["patch_class"] = [
    assign_patch_class(transition_density, porosity)
    for transition_density, porosity in zip(annotations["transition_density"], annotations["porosity"])
]
annotations.to_parquet(OUTPUT_DIR / "patch_annotations.parquet", index=False)

print(f"Transition threshold (75th percentile): {transition_threshold:.6f}")
print(f"High-porosity threshold (75th percentile): {phi_threshold:.6f}")
annotations.groupby(["split", "patch_class"]).size().rename("n_patches")


In [ ]:
sampled_features_by_class: dict[str, list[np.ndarray]] = defaultdict(list)
sampled_meta_by_class: dict[str, list[dict[str, object]]] = defaultdict(list)
seen_per_class: dict[str, int] = defaultdict(int)
channel_rows: list[dict[str, object]] = []


def maybe_reservoir_store(label: str, feature: np.ndarray, meta: dict[str, object]) -> None:
    seen = seen_per_class[label]
    bucket = sampled_features_by_class[label]
    meta_bucket = sampled_meta_by_class[label]
    if len(bucket) < MAX_SAMPLES_PER_CLASS:
        bucket.append(feature.astype(np.float16, copy=False))
        meta_bucket.append(meta)
    else:
        replace_at = int(rng.integers(0, seen + 1))
        if replace_at < MAX_SAMPLES_PER_CLASS:
            bucket[replace_at] = feature.astype(np.float16, copy=False)
            meta_bucket[replace_at] = meta
    seen_per_class[label] += 1


for split_name, loader in {"val": val_loader, "test": test_loader}.items():
    dataset_index = 0
    for batch in tqdm(loader, desc=f"{split_name} latent scan"):
        xct = batch["xct"].to(runtime.device, non_blocking=True)
        mask = batch["mask"].to(runtime.device, non_blocking=True)
        with torch.no_grad():
            mu = encode_mu(runtime.model, xct, mask)

        transition_density = mask_gradient_density(mask).cpu().numpy()
        porosity = mask.float().mean(dim=(1, 2, 3, 4)).cpu().numpy()
        channel_means = mean_latent_channels(mu).cpu().numpy()
        flat_mu = flatten_mu(mu).cpu().numpy()

        for i in range(mu.shape[0]):
            label = assign_patch_class(float(transition_density[i]), float(porosity[i]))
            meta = {
                "split": split_name,
                "dataset_index": dataset_index,
                "patch_class": label,
                "transition_density": float(transition_density[i]),
                "porosity": float(porosity[i]),
            }
            maybe_reservoir_store(label, flat_mu[i], meta)

            row = dict(meta)
            for channel_idx, value in enumerate(channel_means[i]):
                row[f"mu_ch{channel_idx:02d}_mean"] = float(value)
            channel_rows.append(row)
            dataset_index += 1

channel_df = pd.DataFrame(channel_rows)
channel_df.to_parquet(OUTPUT_DIR / "channel_means.parquet", index=False)

sampled_labels = [label for label, meta_rows in sampled_meta_by_class.items() if meta_rows]
sampled_meta = pd.concat(
    [pd.DataFrame(sampled_meta_by_class[label]) for label in sampled_labels],
    ignore_index=True,
)
sampled_features = np.concatenate(
    [np.stack(sampled_features_by_class[label]).astype(np.float32) for label in sampled_labels],
    axis=0,
)
sampled_meta.to_parquet(OUTPUT_DIR / "sampled_meta.parquet", index=False)

print(f"Sampled latent vectors for PCA + UMAP: {sampled_features.shape[0]:,} patches x {sampled_features.shape[1]:,} features")
sampled_meta.groupby("patch_class").size().rename("n_sampled")


In [ ]:
scaler = StandardScaler()
sampled_features_std = scaler.fit_transform(sampled_features)

pca = PCA(n_components=0.95, svd_solver="full", random_state=42)
sampled_features_pca = pca.fit_transform(sampled_features_std)

umap_model = umap.UMAP(
    n_neighbors=UMAP_NEIGHBORS,
    min_dist=UMAP_MIN_DIST,
    metric="euclidean",
    random_state=42,
)
sampled_embedding = umap_model.fit_transform(sampled_features_pca)

sampled_meta["pca_0"] = sampled_features_pca[:, 0]
sampled_meta["pca_1"] = sampled_features_pca[:, 1]
sampled_meta["umap_0"] = sampled_embedding[:, 0]
sampled_meta["umap_1"] = sampled_embedding[:, 1]
sampled_meta.to_parquet(OUTPUT_DIR / "sampled_embedding.parquet", index=False)

binary_mask = sampled_meta["patch_class"].isin(["interior", "high_transition"]).to_numpy()
X_binary = sampled_features_pca[binary_mask]
y_binary = sampled_meta.loc[binary_mask, "patch_class"].to_numpy()

silhouette = silhouette_score(X_binary, y_binary)
cv = StratifiedKFold(n_splits=KNN_FOLDS, shuffle=True, random_state=42)
knn_scores = cross_val_score(
    KNeighborsClassifier(n_neighbors=5),
    X_binary,
    y_binary,
    cv=cv,
    scoring="accuracy",
)

pd.DataFrame({
    "n_sampled": [sampled_features.shape[0]],
    "latent_dim": [sampled_features.shape[1]],
    "pca_components": [sampled_features_pca.shape[1]],
    "pca_variance_retained": [pca.explained_variance_ratio_.sum()],
    "silhouette_interior_vs_transition": [silhouette],
    "knn_accuracy_mean": [knn_scores.mean()],
    "knn_accuracy_std": [knn_scores.std()],
})


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
colors = {
    "interior": "#4C78A8",
    "high_transition": "#E45756",
    "high_porosity": "#72B7B2",
}

for label, group in sampled_meta.groupby("patch_class"):
    axes[0].scatter(group["pca_0"], group["pca_1"], s=6, alpha=0.25, label=label, c=colors.get(label))
    axes[1].scatter(group["umap_0"], group["umap_1"], s=6, alpha=0.25, label=label, c=colors.get(label))

axes[0].set_title("PCA (first two retained components)")
axes[1].set_title("UMAP on PCA-95 features")
for ax in axes:
    ax.legend(loc="best")
    ax.set_xlabel(ax.get_xlabel() or "component 1")
    ax.set_ylabel(ax.get_ylabel() or "component 2")

plt.tight_layout()
plt.show()


In [ ]:
correlation_rows = []
for channel_idx in range(runtime.model.cfg.z_channels):
    correlation_rows.append({
        "channel": channel_idx,
        "pearson_r": channel_df[f"mu_ch{channel_idx:02d}_mean"].corr(channel_df["transition_density"]),
    })

correlation_df = pd.DataFrame(correlation_rows)
correlation_df["abs_pearson_r"] = correlation_df["pearson_r"].abs()
correlation_df = correlation_df.sort_values("abs_pearson_r", ascending=False).reset_index(drop=True)
correlation_df.to_parquet(OUTPUT_DIR / "channel_transition_correlations.parquet", index=False)

display(correlation_df)

plt.figure(figsize=(8, 4))
plt.bar(correlation_df["channel"].astype(str), correlation_df["pearson_r"], color="#4C78A8")
plt.axhline(0.0, color="black", linewidth=1)
plt.title("Per-channel correlation with transition density")
plt.xlabel("latent channel")
plt.ylabel("Pearson r")
plt.tight_layout()
plt.show()


In [ ]:
top_channel = int(correlation_df.iloc[0]["channel"])
top_channel_r = float(correlation_df.iloc[0]["pearson_r"])

print(f"PCA retained {sampled_features_pca.shape[1]} components and {pca.explained_variance_ratio_.sum():.2%} variance.")
print(f"Silhouette score (interior vs high_transition): {silhouette:.4f}")
print(f"5-NN accuracy (cross-validated): {knn_scores.mean():.4f} +/- {knn_scores.std():.4f}")
print(f"Strongest channel/transition correlation: channel {top_channel} with r={top_channel_r:+.4f}")

if knn_scores.mean() > 0.65 and silhouette > 0.05:
    print("Interpretation: transition-heavy patches are separable in mu, so decoder-side underuse is the more likely bottleneck.")
    print("If R03a also shows a strong auxiliary-decoder gain, both notebooks point to decoder-side R04 interventions.")
else:
    print("Interpretation: transition-heavy patches are only weakly separable in mu, so the latent bottleneck remains plausible.")
    print("Combined with a weak R03a auxiliary gain, this points to increasing z_channels and/or reducing downsampling in R04.")
